# VentureScope — Exploratory Data Analysis
## Global Startup Ecosystem Analytics

**Dataset:** Crunchbase StartUp Investments (Kaggle)  
**Source:** https://www.kaggle.com/datasets/justinas/startup-investments  
**Records:** 379,378 processed records (130K companies, 52K funding rounds, 74K investments)  

---

In [ ]:
# Import libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings

warnings.filterwarnings('ignore')
sns.set_theme(style='darkgrid', palette='viridis')
plt.rcParams['figure.figsize'] = (12, 6)
plt.rcParams['figure.dpi'] = 100

print('Libraries loaded successfully ✓')

## 1. Dataset Loading

In [ ]:
# Load processed datasets
companies = pd.read_csv('../data/processed/companies.csv', low_memory=False)
funding = pd.read_csv('../data/processed/funding_rounds.csv', low_memory=False)
investors = pd.read_csv('../data/processed/investors.csv', low_memory=False)
offices = pd.read_csv('../data/processed/offices.csv', low_memory=False)
acquisitions = pd.read_csv('../data/processed/acquisitions.csv', low_memory=False)

print(f'Companies:       {len(companies):>10,} rows')
print(f'Funding Rounds:  {len(funding):>10,} rows')
print(f'Investors:       {len(investors):>10,} rows')
print(f'Offices:         {len(offices):>10,} rows')
print(f'Acquisitions:    {len(acquisitions):>10,} rows')
print(f'{"─"*35}')
total = len(companies) + len(funding) + len(investors) + len(offices) + len(acquisitions)
print(f'TOTAL:           {total:>10,} rows')

## 2. Data Schema Exploration

In [ ]:
# Companies schema
print('COMPANIES TABLE')
print('=' * 60)
print(f'Shape: {companies.shape}')
print(f'\nColumn types:')
print(companies.dtypes)
print(f'\nMissing values (top 10):')
print(companies.isnull().sum().sort_values(ascending=False).head(10))

In [ ]:
# First 5 rows
companies.head()

In [ ]:
# Funding rounds schema
print('FUNDING ROUNDS TABLE')
print('=' * 60)
print(f'Shape: {funding.shape}')
print(f'\nKey statistics:')
print(funding[['raised_amount_usd', 'pre_money_valuation_usd', 'post_money_valuation_usd']].describe())

## 3. Company Distribution Analysis

In [ ]:
# Company status distribution
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Status pie chart
status_counts = companies['status'].value_counts()
colors = ['#10b981', '#3b82f6', '#ef4444', '#8b5cf6']
axes[0].pie(status_counts.values, labels=status_counts.index,
            autopct='%1.1f%%', colors=colors, startangle=90,
            textprops={'fontsize': 11})
axes[0].set_title('Company Status Distribution', fontsize=14, fontweight='bold')

# Founded year distribution
founded = companies['founded_year'].dropna()
founded = founded[(founded >= 1995) & (founded <= 2013)]
axes[1].hist(founded, bins=19, color='#3b82f6', edgecolor='white', alpha=0.8)
axes[1].set_title('Companies Founded per Year', fontsize=14, fontweight='bold')
axes[1].set_xlabel('Year')
axes[1].set_ylabel('Number of Companies')

plt.tight_layout()
plt.savefig('../report/figures/company_distribution.png', dpi=150, bbox_inches='tight')
plt.show()

## 4. Sector Analysis

In [ ]:
# Top 15 sectors by company count
sector_counts = companies['sector'].value_counts().head(15)

fig, ax = plt.subplots(figsize=(12, 7))
bars = ax.barh(range(len(sector_counts)), sector_counts.values, color='#3b82f6', edgecolor='white')
ax.set_yticks(range(len(sector_counts)))
ax.set_yticklabels(sector_counts.index, fontsize=11)
ax.set_xlabel('Number of Companies', fontsize=12)
ax.set_title('Top 15 Sectors by Company Count', fontsize=14, fontweight='bold')
ax.invert_yaxis()

# Add value labels
for bar, val in zip(bars, sector_counts.values):
    ax.text(val + 200, bar.get_y() + bar.get_height()/2,
            f'{val:,}', va='center', fontsize=10)

plt.tight_layout()
plt.savefig('../report/figures/sector_distribution.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Sector funding analysis
funded = companies[companies['funding_total_usd'] > 0]
sector_funding = funded.groupby('sector')['funding_total_usd'].agg(['sum', 'mean', 'median', 'count'])
sector_funding.columns = ['Total Funding', 'Mean Funding', 'Median Funding', 'Company Count']
sector_funding = sector_funding.sort_values('Total Funding', ascending=False)

# Format as readable
display_df = sector_funding.copy()
for col in ['Total Funding', 'Mean Funding', 'Median Funding']:
    display_df[col] = display_df[col].apply(lambda x: f'${x/1e9:.2f}B' if x >= 1e9 else f'${x/1e6:.1f}M')
display_df['Company Count'] = display_df['Company Count'].apply(lambda x: f'{x:,}')

display_df

## 5. Geographic Analysis

In [ ]:
# Top 20 countries
country_counts = companies[companies['country'] != 'Unknown']['country'].value_counts().head(20)

fig, ax = plt.subplots(figsize=(12, 8))
colors = plt.cm.Blues(np.linspace(0.4, 0.9, len(country_counts)))[::-1]
bars = ax.barh(range(len(country_counts)), country_counts.values, color=colors, edgecolor='white')
ax.set_yticks(range(len(country_counts)))
ax.set_yticklabels(country_counts.index, fontsize=11)
ax.set_xlabel('Number of Companies', fontsize=12)
ax.set_title('Top 20 Countries by Startup Count', fontsize=14, fontweight='bold')
ax.invert_yaxis()

for bar, val in zip(bars, country_counts.values):
    ax.text(val + 100, bar.get_y() + bar.get_height()/2,
            f'{val:,}', va='center', fontsize=10)

plt.tight_layout()
plt.savefig('../report/figures/country_distribution.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Regional analysis
region_counts = companies[companies['world_region'] != 'Other']['world_region'].value_counts()

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Region by company count
colors_region = ['#3b82f6', '#8b5cf6', '#10b981', '#f59e0b', '#ef4444']
axes[0].pie(region_counts.values, labels=region_counts.index,
            autopct='%1.1f%%', colors=colors_region, startangle=90,
            textprops={'fontsize': 10})
axes[0].set_title('Companies by Region', fontsize=13, fontweight='bold')

# Region by funding
region_funding = funded.groupby('world_region')['funding_total_usd'].sum().sort_values(ascending=True)
region_funding = region_funding[region_funding.index != 'Other']
axes[1].barh(region_funding.index, region_funding.values / 1e9,
             color=colors_region[:len(region_funding)][::-1])
axes[1].set_xlabel('Total Funding ($B)')
axes[1].set_title('Total Funding by Region', fontsize=13, fontweight='bold')

plt.tight_layout()
plt.savefig('../report/figures/region_analysis.png', dpi=150, bbox_inches='tight')
plt.show()

## 6. Funding Analysis

In [ ]:
# Funding over time
funding['funding_date'] = pd.to_datetime(funding['funded_at'], errors='coerce')
funding['fund_year'] = funding['funding_year']
yearly = funding[funding['fund_year'].between(1995, 2013)].groupby('fund_year').agg(
    total_funding=('raised_amount_usd', 'sum'),
    deal_count=('funding_round_id', 'count'),
    avg_deal=('raised_amount_usd', 'mean'),
).reset_index()

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Total funding over time
axes[0].fill_between(yearly['fund_year'], yearly['total_funding'] / 1e9,
                     alpha=0.3, color='#3b82f6')
axes[0].plot(yearly['fund_year'], yearly['total_funding'] / 1e9,
             color='#3b82f6', linewidth=2, marker='o', markersize=4)
axes[0].set_xlabel('Year')
axes[0].set_ylabel('Total Funding ($B)')
axes[0].set_title('Total Startup Funding Over Time', fontsize=13, fontweight='bold')

# Deal count over time
axes[1].bar(yearly['fund_year'], yearly['deal_count'],
            color='#8b5cf6', edgecolor='white', alpha=0.8)
axes[1].set_xlabel('Year')
axes[1].set_ylabel('Number of Deals')
axes[1].set_title('Deal Count Over Time', fontsize=13, fontweight='bold')

plt.tight_layout()
plt.savefig('../report/figures/funding_timeline.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Funding by stage
stage_order = ['Angel', 'Venture', 'Series A', 'Series B', 'Series C+', 'Private Equity', 'Other']
stage_funding = funding.groupby('funding_stage')['raised_amount_usd'].agg(['sum', 'count', 'mean', 'median'])
stage_funding.columns = ['Total', 'Count', 'Mean', 'Median']
stage_funding = stage_funding.reindex([s for s in stage_order if s in stage_funding.index])

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Total by stage
colors_stage = ['#f59e0b', '#3b82f6', '#10b981', '#8b5cf6', '#ef4444', '#06b6d4', '#71717a']
axes[0].barh(stage_funding.index, stage_funding['Total'] / 1e9,
             color=colors_stage[:len(stage_funding)])
axes[0].set_xlabel('Total Funding ($B)')
axes[0].set_title('Total Funding by Stage', fontsize=13, fontweight='bold')

# Deal count by stage
axes[1].barh(stage_funding.index, stage_funding['Count'],
             color=colors_stage[:len(stage_funding)])
axes[1].set_xlabel('Number of Deals')
axes[1].set_title('Deal Count by Stage', fontsize=13, fontweight='bold')

plt.tight_layout()
plt.savefig('../report/figures/funding_by_stage.png', dpi=150, bbox_inches='tight')
plt.show()

## 7. Investor Analysis

In [ ]:
# Top 20 investors by deal count
inv_summary = pd.read_csv('../data/processed/investor_summary.csv')
top20 = inv_summary.nlargest(20, 'deal_count')

fig, ax = plt.subplots(figsize=(12, 8))
bars = ax.barh(range(len(top20)), top20['deal_count'].values,
               color=plt.cm.Blues(np.linspace(0.4, 0.9, len(top20)))[::-1])
ax.set_yticks(range(len(top20)))
ax.set_yticklabels(top20['investor_name'].values, fontsize=10)
ax.set_xlabel('Number of Deals', fontsize=12)
ax.set_title('Top 20 Most Active Investors', fontsize=14, fontweight='bold')
ax.invert_yaxis()

for bar, val in zip(bars, top20['deal_count'].values):
    ax.text(val + 5, bar.get_y() + bar.get_height()/2,
            f'{val:,.0f}', va='center', fontsize=10)

plt.tight_layout()
plt.savefig('../report/figures/top_investors.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Investor x Sector heatmap
inv_sector = pd.read_csv('../data/processed/investor_sector_matrix.csv')
pivot = inv_sector.pivot_table(index='investor_name', columns='sector',
                               values='deal_count', fill_value=0)
pivot['total'] = pivot.sum(axis=1)
pivot = pivot.nlargest(15, 'total').drop(columns='total')
pivot = pivot.loc[:, pivot.sum() > 3]

fig, ax = plt.subplots(figsize=(14, 8))
sns.heatmap(pivot, annot=True, fmt='g', cmap='Blues',
            linewidths=0.5, linecolor='white', ax=ax,
            cbar_kws={'label': 'Deal Count'})
ax.set_title('Investor × Sector Deal Heatmap (Top 15 Investors)', fontsize=14, fontweight='bold')
ax.set_xlabel('Sector')
ax.set_ylabel('Investor')
plt.xticks(rotation=45, ha='right')

plt.tight_layout()
plt.savefig('../report/figures/investor_sector_heatmap.png', dpi=150, bbox_inches='tight')
plt.show()

## 8. Success & Failure Analysis

In [ ]:
# Success rate by sector
sector_status = companies.groupby('sector')['status'].value_counts().unstack(fill_value=0)
sector_status['total'] = sector_status.sum(axis=1)
sector_status = sector_status[sector_status['total'] >= 500]  # Min 500 companies

if 'acquired' in sector_status.columns and 'ipo' in sector_status.columns:
    sector_status['success'] = sector_status['acquired'] + sector_status['ipo']
elif 'acquired' in sector_status.columns:
    sector_status['success'] = sector_status['acquired']
else:
    sector_status['success'] = 0

sector_status['success_rate'] = (sector_status['success'] / sector_status['total'] * 100).round(1)
sector_status = sector_status.sort_values('success_rate', ascending=True)

fig, ax = plt.subplots(figsize=(12, 7))
colors = plt.cm.RdYlGn(np.linspace(0.2, 0.8, len(sector_status)))
ax.barh(sector_status.index, sector_status['success_rate'],
        color=colors, edgecolor='white')
ax.set_xlabel('Exit Success Rate (%)', fontsize=12)
ax.set_title('Startup Exit Success Rate by Sector', fontsize=14, fontweight='bold')
ax.axvline(x=sector_status['success_rate'].mean(), color='red',
           linestyle='--', alpha=0.7, label=f'Average: {sector_status["success_rate"].mean():.1f}%')
ax.legend()

plt.tight_layout()
plt.savefig('../report/figures/success_rate.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Funding distribution (log scale)
funding_vals = funded['funding_total_usd'].dropna()
funding_vals = funding_vals[funding_vals > 0]

fig, ax = plt.subplots(figsize=(12, 5))
ax.hist(np.log10(funding_vals), bins=50, color='#3b82f6',
        edgecolor='white', alpha=0.8)
ax.set_xlabel('Log₁₀(Total Funding USD)', fontsize=12)
ax.set_ylabel('Number of Companies', fontsize=12)
ax.set_title('Distribution of Startup Funding (Log Scale)', fontsize=14, fontweight='bold')

# Add labels for scale reference
for val, label in [(3, '$1K'), (5, '$100K'), (6, '$1M'), (7, '$10M'), (8, '$100M'), (9, '$1B')]:
    ax.axvline(x=val, color='gray', linestyle=':', alpha=0.3)
    ax.text(val, ax.get_ylim()[1]*0.95, label, ha='center', fontsize=9, color='gray')

plt.tight_layout()
plt.savefig('../report/figures/funding_distribution.png', dpi=150, bbox_inches='tight')
plt.show()

## 9. Key Insights Summary

### Findings:

1. **US Dominance:** The United States accounts for ~40% of all startups in the dataset
2. **Sector Leaders:** SaaS/Software and Internet/Web dominate by both company count and funding
3. **Power Law Distribution:** Funding follows a heavy-tailed power law — most startups raise small amounts while a few raise billions
4. **Low Exit Rates:** Only ~8% of startups achieve exits (acquisition + IPO)
5. **Investor Specialization:** Top investors show clear sector preferences (visible in heatmap)
6. **Temporal Trends:** Startup creation and funding accelerated from 2005 onwards, peaking around 2012-2013

In [ ]:
# Summary statistics
print('=' * 60)
print('DATASET SUMMARY')
print('=' * 60)
print(f'Total Companies:     {len(companies):>10,}')
print(f'Funded Companies:    {len(funded):>10,}')
print(f'Total Funding:       ${funded["funding_total_usd"].sum()/1e9:>9.1f}B')
print(f'Median Funding:      ${funded["funding_total_usd"].median()/1e6:>9.2f}M')
print(f'Countries:           {companies["country"].nunique():>10}')
print(f'Sectors:             {companies["sector"].nunique():>10}')
print(f'Funding Rounds:      {len(funding):>10,}')
print(f'Unique Investors:    {investors["investor_name"].nunique():>10,}')
print(f'Acquisitions:        {len(acquisitions):>10,}')
print(f'Office Locations:    {len(offices):>10,}')
print('=' * 60)